In [ ]:
%load_ext autoreload
%autoreload 2

from spatial_tcr.utils import setup_plotting, switch_cwd_to_root

switch_cwd_to_root()

figure_dir = "figures/revision/supplement-extra"
setup_plotting(figure_dir, display_dpi=300, save_dpi=300)

import os

import scanpy as sc

from spatial_tcr.colors import cell_to_group, colors_sub
from spatial_tcr.plotting import (
    plot_cell_types_per_domain_stacked,
    plot_tcell_infiltrate_per_domain_stacked,
)

## Load data

In [ ]:
data_dir = "data/xenium/processed"
path = f"{data_dir}/08.2-kidney_tcr_clonal_clusters_peri_glom_annot.h5ad"
adata = sc.read_h5ad(path)

# remove control samples
adata = adata[adata.obs["condition"] == "ANCA"].copy()
adata

In [ ]:
samples = adata.obs["sample"].unique()
ad_sub = adata[adata.obs["sample"].isin(samples[0:1])].copy()
sc.pl.spatial(ad_sub, color=["tcell_infiltrate"], spot_size=10)

## Characterization in terms of cell phenotypes

In [ ]:
adata.obs.columns

In [ ]:
adata.obs["in_glom"].value_counts()

In [ ]:
adata.obs["tcell_infiltrate"].value_counts()

In [ ]:
# make sure overlap with gloms is resolved
adata.obs["domain"] = adata.obs["tcell_infiltrate"].astype(str)
adata.obs.loc[adata.obs["in_glom"], "domain"] = "glom"
adata.obs.loc[adata.obs["domain"] == "no infiltrate", "domain"] = "no\ninfiltrate"

ad_sub = adata[adata.obs["sample"].isin(samples[0:1])].copy()
sc.pl.spatial(ad_sub, color=["domain"], spot_size=10)

In [ ]:
adata.obs["cell_type_l0"] = adata.obs["cell_type_l2"].astype(str).replace(cell_to_group)

In [ ]:
ct_counts = (
    adata.obs.groupby(["domain"], observed=True)["cell_type_l0"]
    .value_counts(normalize=True)
    .unstack()
    .fillna(0)
)
ct_counts = ct_counts.loc[["infiltrate", "no\ninfiltrate", "glom"]]
ct_counts

In [ ]:
fig, ax = plot_cell_types_per_domain_stacked(
    ct_counts,
    title="Cell Type Proportions by Domain",
    colors_dict=colors_sub,
    save_path=os.path.join(figure_dir, "cell_type_proportions_by_domain.pdf"),
)

## Characterization in terms of clonotypes

In [ ]:
# Change category name using .cat.rename_categories for categorical dtype
adata.obs["tcell_infiltrate"] = adata.obs["tcell_infiltrate"].cat.rename_categories(
    {
        cat: "no\ninfiltrate" if cat == "no infiltrate" else cat
        for cat in adata.obs["tcell_infiltrate"].cat.categories
    }
)

clone_counts = (
    adata.obs.loc[adata.obs["avbv_clone"].notna()]
    .groupby(["tcell_infiltrate"], observed=True)["avbv_clone"]
    .value_counts(normalize=False)
    .unstack()
    .fillna(0)
)
# Order columns by cumulative abundance (sum across all rows), descending
clone_counts = clone_counts.loc[
    :, clone_counts.sum(axis=0).sort_values(ascending=False).index
]
clone_counts = clone_counts.loc[["infiltrate", "no\ninfiltrate"]]

In [ ]:
from spatial_tcr.plotting import plot_clone_counts_per_domain_stacked

fig, ax = plot_clone_counts_per_domain_stacked(
    clone_counts,
    top_n=10,
    normalize=True,  # or False for counts
    title="Clone Proportions by Domain",
    ylim_max=0.2,
    save_path=os.path.join(figure_dir, "clone_proportions_by_domain.pdf"),
)

### Check per sample

In [ ]:
adata.obs["sample_short"] = (
    adata.obs["sample"]
    .astype(str)
    .replace({s: f"S{i}" for i, s in enumerate(adata.obs["sample"].unique())})
)

for sample in adata.obs["sample_short"].unique():
    ad_sub = adata[adata.obs["sample_short"] == sample].copy()
    obs = ad_sub.obs.loc[ad_sub.obs["avbv_clone"].notna()]
    if "infiltrate" not in obs["tcell_infiltrate"].unique():
        continue
    clone_counts = (
        obs.groupby(["tcell_infiltrate"], observed=True)["avbv_clone"]
        .value_counts(normalize=False)
        .unstack()
        .fillna(0)
    )
    # Order columns by cumulative abundance (sum across all rows), descending
    clone_counts = clone_counts.loc[
        :, clone_counts.sum(axis=0).sort_values(ascending=False).index
    ]
    clone_counts = clone_counts.loc[["infiltrate", "no\ninfiltrate"]]

    fig, ax = plot_clone_counts_per_domain_stacked(
        clone_counts,
        top_n=10,
        normalize=True,  # or False for counts
        title=f"Clone Proportions by Domain\n(sample {sample})",
        ylim_max=0.2,
    )

## Characterize localization

In [ ]:
adata.obs.columns

In [ ]:
adata.obs["tcell_infiltrate"].value_counts()

In [ ]:
adata.obs["domain_new"] = "tubulointerstitial"
adata.obs.loc[adata.obs["in_glom"], "domain_new"] = "glomerular"
adata.obs.loc[adata.obs["in_peri_glom"], "domain_new"] = "periglomerular"

adata.obs["domain_new"].value_counts()

In [ ]:
plot_tcell_infiltrate_per_domain_stacked(
    adata,
    title="T cell infiltrate proportions per domain",
    figsize=(1.5, 3),
    exclude_domains=["glomerular"],
    save_path=os.path.join(figure_dir, "tcell_infiltrate_proportions_per_domain.pdf"),
)